The Setup:  Imagine two people working on a project together. Each person
could naturally be either a leader or a follower, but they don't know which role
the other person prefers. Through repeated interactions, they need to figure out
who should lead and who should follow.

The Twist:  Each person forms beliefs about the other person's role, but also
thinks about what the other person believes about their own role. This creates
recursive reasoning: ”I think you think I'm a leader, so I should act like one.”

The Goal:  Show how stable leader-follower relationships emerge through this
recursive belief updating process.

Currently this notebook just has payoff 1 in recursive model.
To show institutional convergence, we need to break symmetry:

  Option 1: Add slight role biases
  - Alice has 55% prior for LEADER
  - Bob has 55% prior for FOLLOWER
  - Recursive reasoning amplifies these small differences

  Option 2: Add observations/learning
  - Agents observe outcomes and update beliefs
  - Over iterations, roles crystallize

  Option 3: Add a coordination signal
  - One agent gets a slight "first mover" advantage

In [14]:
# Import dependencies
from functools import cache
import jax
import jax.numpy as np
from memo import memo
from enum import Enum
import matplotlib.pyplot as plt

In [15]:
from enum import IntEnum

class ACTIONS(IntEnum):
    ASSERT = 0 
    DEFER = 1

class ROLES(IntEnum):
    LEADER = 0
    FOLLOWER = 1

# Payoff matrix: [my_action, other_action] -> reward
# Both asserting (conflict) is bad, both deferring (indecision) is bad
# One asserts, one defers (coordination) is good
PAYOFF_MATRIX = np.array([
    [0.0, 1.0],  # If I assert: get 0 if other asserts, get 1 if other defers
    [0.5, 0.5]   # If I defer: get 0.5 regardless (safe but suboptimal)
])

@jax.jit
def individual_payoff(action_alice, action_bob):
    return PAYOFF_MATRIX[action_alice, action_bob]

@jax.jit
def collective_payoff(action_alice, action_bob):
    return PAYOFF_MATRIX[action_alice, action_bob] + PAYOFF_MATRIX[action_bob, action_alice]

# Rationality parameter for action selection
@jax.jit
def beta():
    return 3.0

In [16]:
# Random game where agents randomly choose an action
@memo 
def random_game[action: ACTIONS]():
    alice: chooses(action_alice in ACTIONS, wpp=1)
    bob: chooses(action_bob in ACTIONS, wpp=1)
    return E[collective_payoff(alice.action_alice, bob.action_bob)]
random_game()

Array([1.], dtype=float32)

In [17]:
# Naive game where each agent has random role
@memo 
def naive_game[role: ROLES, action: ACTIONS]():
    alice: chooses(role in ROLES, wpp = 1)
    alice: chooses(action_alice in ACTIONS, wpp = role == action_alice)
    bob: chooses(role in ROLES, wpp = 1)
    bob: chooses(action_bob in ACTIONS, wpp = role == action_bob)
    return E[collective_payoff(alice.action_alice, bob.action_bob)]
naive_game()

Array([[1.]], dtype=float32)

In [18]:
# Hierarchial game where each agent prefers a different role
@memo
def hierarchial_game[role: ROLES, action: ACTIONS]():
    alice: chooses(role in ROLES, wpp = role == 0)  # Prefer LEADER (0)
    alice: chooses(action_alice in ACTIONS, wpp = role == action_alice)
    bob: chooses(role in ROLES, wpp = role == 1)  # Prefer FOLLOWER (1)
    bob: chooses(action_bob in ACTIONS, wpp = role == action_bob)
    return E[collective_payoff(alice.action_alice, bob.action_bob)]
hierarchial_game()

Array([[1.5]], dtype=float32)

In [19]:
# Rational game where Alice models Bob's behavior and acts strategically
@memo
def one_sided_rational_game[role: ROLES, action: ACTIONS]():
    # Bob's actual role and action
    bob: chooses(role_bob in ROLES, wpp = 1)
    bob: chooses(action_bob in ACTIONS, wpp = role_bob == action_bob)
    
    # Alice's role
    alice: chooses(role_alice in ROLES, wpp = 1)
    
    # Alice models what Bob will do
    alice: thinks[
        bob: chooses(role_bob in ROLES, wpp = 1),
        bob: chooses(action_bob in ACTIONS, wpp = role_bob == action_bob)
    ]
    
    # Alice chooses action to maximize EXPECTED payoff over her beliefs about Bob
    alice: chooses(action_alice in ACTIONS,
                   to_maximize = E[individual_payoff(action_alice, bob.action_bob)] * beta())
    
    return E[collective_payoff(alice.action_alice, bob.action_bob)]
one_sided_rational_game()

Array([[1.]], dtype=float32)

In [20]:
# Mutual rational game where both Alice and Bob model each other
@memo
def mutual_rational_game[role: ROLES, action: ACTIONS]():
    # Alice's role
    alice: chooses(role_alice in ROLES, wpp = 1)
    
    # Bob's role
    bob: chooses(role_bob in ROLES, wpp = 1)
    
    # Alice models Bob as naive (acts according to role)
    alice: thinks[
        bob: chooses(role_bob in ROLES, wpp = 1),
        bob: chooses(action_bob in ACTIONS, wpp = role_bob == action_bob)
    ]
    
    # Bob models Alice as naive (acts according to role)
    bob: thinks[
        alice: chooses(role_alice in ROLES, wpp = 1),
        alice: chooses(action_alice in ACTIONS, wpp = role_alice == action_alice)
    ]
    
    # Alice chooses rationally based on her model of Bob
    alice: chooses(action_alice in ACTIONS,
                   to_maximize = E[individual_payoff(action_alice, bob.action_bob)] * beta())
    
    # Bob chooses rationally based on his model of Alice
    bob: chooses(action_bob in ACTIONS,
                 to_maximize = E[individual_payoff(action_bob, alice.action_alice)] * beta())
    
    return E[collective_payoff(alice.action_alice, bob.action_bob)]
mutual_rational_game()

Array([[1.]], dtype=float32)

In [21]:
# Recursive reasoning game with depth parameter
@memo
def recursive_game[role: ROLES, action: ACTIONS](depth):
    # Both agents have roles
    alice: chooses(role_alice in ROLES, wpp = 1)
    bob: chooses(role_bob in ROLES, wpp = 1)
    
    # Alice models Bob at depth-1 (or as naive if depth=0)
    alice: thinks[
        bob: chooses(role_bob in ROLES, wpp = 1),
        bob: chooses(action_bob in ACTIONS, 
                     wpp = recursive_game[role_bob, action_bob](depth - 1) if depth > 0 else (role_bob == action_bob))
    ]
    
    # Bob models Alice at depth-1 (or as naive if depth=0)
    bob: thinks[
        alice: chooses(role_alice in ROLES, wpp = 1),
        alice: chooses(action_alice in ACTIONS,
                       wpp = recursive_game[role_alice, action_alice](depth - 1) if depth > 0 else (role_alice == action_alice))
    ]
    
    # Both choose actions: rationally (softmax) if depth > 0, naively if depth = 0
    alice: chooses(action_alice in ACTIONS,
                   wpp = exp(beta() * E[individual_payoff(action_alice, bob.action_bob)]) if depth > 0 else (role_alice == action_alice))
    
    bob: chooses(action_bob in ACTIONS,
                 wpp = exp(beta() * E[individual_payoff(action_bob, alice.action_alice)]) if depth > 0 else (role_bob == action_bob))
    
    return E[collective_payoff(alice.action_alice, bob.action_bob)]

# Test at different depths
for d in range(5):
    result = recursive_game(d)
    print(f'Depth {d}:')
    print(result)

Depth 0:
[[1.]]
Depth 1:
[[1.]]
Depth 2:
[[1.]]
Depth 3:
[[1.]]
Depth 4:
[[1.]]
